# KaraokeForge on Kaggle GPU

Run this notebook in a Kaggle Notebook with **GPU** enabled. It starts the real `worker/` service from the GitHub repo and exposes it with a free Cloudflare Quick Tunnel.

Kaggle currently documents free NVIDIA P100 access with a weekly GPU quota around 30 hours. Cloudflare Quick Tunnels are free for testing/development and provide a temporary `trycloudflare.com` HTTPS URL.

In [ ]:
!rm -rf /kaggle/working/KaraokeForge
!git clone -q https://github.com/omaparekh-ux/KaraokeForge.git /kaggle/working/KaraokeForge
%cd /kaggle/working/KaraokeForge/worker


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -r requirements.txt


In [ ]:
import os
os.environ["JOBS_DIR"] = "/kaggle/working/karaoke_jobs"
os.environ["JOB_DB"] = "/kaggle/working/karaoke_jobs/karaokeforge.db"
os.environ["FRONTEND_ORIGINS"] = "*"
os.environ["DEMUCS_DEVICE"] = "cuda"
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE"] = "float16"
os.environ["WHISPER_MODEL"] = "small"
os.environ["DEMUCS_MODEL"] = "htdemucs"
os.environ["MAX_CONCURRENT_JOBS"] = "1"
os.environ["MAX_UPLOAD_MB"] = "250"
print("Configured KaraokeForge worker")


In [ ]:
!nvidia-smi -L


In [ ]:
import subprocess, time, os
log_path = "/kaggle/working/karaokeforge-worker.log"
log = open(log_path, "w")
proc = subprocess.Popen(["python","-m","uvicorn","app:app","--host","0.0.0.0","--port","8000"], cwd="/kaggle/working/KaraokeForge/worker", env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)
time.sleep(4)
print("Worker PID:", proc.pid)
print(open(log_path).read()[-2000:])


In [ ]:
import subprocess, time, re, os
cloudflared = "/kaggle/working/cloudflared"
if not os.path.exists(cloudflared):
    subprocess.run(["bash","-lc","curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared"], check=True)
log2 = open("/kaggle/working/cloudflared.log", "w")
tunnel = subprocess.Popen([cloudflared,"tunnel","--url","http://127.0.0.1:8000"], stdout=log2, stderr=subprocess.STDOUT)
time.sleep(5)
text = open("/kaggle/working/cloudflared.log").read()
match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", text)
if not match:
    print(text[-5000:])
    raise RuntimeError("Tunnel URL not found. Re-run this cell.")
WORKER_URL = match.group(0)
print("KaraokeForge worker URL:")
print(WORKER_URL)


In [ ]:
import requests
print(requests.get(WORKER_URL + "/health", timeout=20).json())
print("\nPaste this URL into KaraokeForge → Processing engine → Save worker:")
print(WORKER_URL)


## Keep the session alive

This is a free, temporary worker. Keep the Kaggle session running while processing songs. When the session ends, the worker URL disappears and generated files are lost.

The worker itself is the same FastAPI implementation in `worker/`; the notebook only provides temporary GPU compute.